In [0]:
from delta import configure_spark_with_delta_pip, DeltaTable
from pyspark.sql import SparkSession

In [0]:
builder = (SparkSession.builder
           .appName("time-travel-delta-table")
           .master("spark://spark-master:7077")
           .config("spark.executor.memory", "512m")
           .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
           .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [0]:
%load_ext sparksql_magic
%config SparkSql.limit=20

In [0]:
%%sparksql
DESCRIBE DETAIL delta.`/opt/workspace/data/delta_lake/netflix_titles`

In [0]:
df = spark.read.format("delta").option("versionAsOf", 1).load("/opt/workspace/data/delta_lake/netflix_titles")
df.show(5)

In [0]:
%%sparksql
SELECT * FROM delta.`/opt/workspace/data/delta_lake/netflix_titles` VERSION AS OF 1 LIMIT 3;

In [0]:
deltaTable = DeltaTable.forPath(spark, "/opt/workspace/data/delta_lake/netflix_titles")  # path-based tables, or
deltaTable.restoreToVersion(3) # restore table to oldest version

In [0]:
%%sparksql
RESTORE TABLE delta.`/opt/workspace/data/delta_lake/netflix_titles` TO VERSION AS OF 3;

In [0]:
%%sparksql
DESCRIBE HISTORY "/opt/workspace/data/delta_lake/netflix_titles"

In [0]:
spark.stop()